In [ ]:
import sys
import os
import json
import pandas as pd
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report

import joblib

sys.path.append(r"C:\Users\germa\Desktop\Ejercicios\Agente-ML-IA-FugaClientes-")
from data.repositorio_cliente import generate_random_customer, get_customer_by_id

db = [generate_random_customer() for i in range (500)]
dataf = pd.DataFrame(db)
# print(json.dumps(db, indent=4, default=str))
print(dataf)

     customer_id signup_date last_login_date   plan  monthly_fee  \
0           3987  2025-03-07      2026-06-02  basic        13.39   
1           4598  2025-12-07      2025-12-30  basic        17.82   
2           4192  2022-11-23      2026-02-05    pro        28.19   
3           5301  2024-06-10      2025-11-13    pro        25.77   
4           4057  2025-09-06      2025-06-19    pro        46.03   
..           ...         ...             ...    ...          ...   
495         9447  2021-09-06      2026-03-13    pro         7.07   
496         5842  2022-10-04      2026-05-10    pro        46.68   
497         9403  2021-02-17      2025-06-26    pro        20.13   
498         5020  2024-11-19      2025-08-13    pro        36.70   
499         2865  2025-11-13      2025-10-09    pro        10.35   

     support_tickets  usage_minutes country  is_active  
0                 20            959      US          1  
1                 13           1842      ES          0  
2           

In [2]:
def preprocesado_datos(dataf):
    today = datetime.today()
    # dataf[["signup_year","signup_M","signup_Day"]] = dataf["signup_date"].str.split('-', expand=True)
    # dataf[["last_login_year","last_login_M","last_login_Day"]] = dataf["last_login_date"].str.split('-', expand=True)

    dataf["signup_date"] = pd.to_datetime(dataf["signup_date"])
    dataf["days_since_signup"] = (today - dataf["signup_date"]).dt.days

    # ---- last_login_date ----
    dataf["last_login_date"] = pd.to_datetime(dataf["last_login_date"])
    dataf["days_since_last_login"] = (today - dataf["last_login_date"]).dt.days


    # col_obj = dataf.select_dtypes(include=object).columns
    # dataf[col_obj] = dataf[col_obj].astype('category')

    dataf = dataf.drop(columns=["signup_date","last_login_date","customer_id"])
    return dataf
    
data_pre = preprocesado_datos(dataf)
data_pre.head(5)


,plan,monthly_fee,support_tickets,usage_minutes,country,is_active,days_since_signup,days_since_last_login
0,basic,13.39,20,959,US,1,466,14
1,basic,17.82,13,1842,ES,0,191,168
2,pro,28.19,9,628,FR,1,1301,131
3,pro,25.77,4,1520,US,1,736,215
4,pro,46.03,7,1118,US,1,283,362


In [3]:
y = data_pre["is_active"]
X = data_pre.drop(columns=["is_active"])
colums_cat = X.select_dtypes(exclude="number").columns
colums_cat

Index(['plan', 'country'], dtype='object')

In [4]:
from sklearn.impute import KNNImputer
import sklearn.impute as skl_imp
import numpy as np

# data_dummies = pd.get_dummies(X,drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

X_train_cat_encoded = encoder.fit_transform(X_train[colums_cat])
X_test_cat_encoded = encoder.transform(X_test[colums_cat])

nombres_dummies = encoder.get_feature_names_out(colums_cat)
df_train_cat = pd.DataFrame(X_train_cat_encoded, columns=nombres_dummies, index=X_train.index)
df_test_cat = pd.DataFrame(X_test_cat_encoded, columns=nombres_dummies, index=X_test.index)

X_train_num = X_train.drop(columns=colums_cat)
X_test_num = X_test.drop(columns=colums_cat)

X_train_final = pd.concat([X_train_num, df_train_cat], axis=1)
X_test_final = pd.concat([X_test_num, df_test_cat], axis=1)


imputer = SimpleImputer(strategy="mean")

X_train_imputado = imputer.fit_transform(X_train_final)
X_test_imputado = imputer.transform(X_test_final)
imputer.feature_names_in_ = X_train_final.columns.values

# MODELO

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train_imputado, y_train)

# EVALUACION

predictions = model.predict(X_test_imputado)
print(classification_report(y_test, predictions))


              precision    recall  f1-score   support

           0       0.95      0.83      0.88        46
           1       0.87      0.96      0.91        54

    accuracy                           0.90       100
   macro avg       0.91      0.89      0.90       100
weighted avg       0.91      0.90      0.90       100



In [5]:
X_test_final

,monthly_fee,support_tickets,usage_minutes,days_since_signup,days_since_last_login,plan_basic,plan_enterprise,plan_pro,country_AR,country_ES,country_FR,country_MX,country_US
361,11.10,11,1180,1926,381,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
73,41.67,16,270,1819,134,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
374,29.05,11,468,1302,182,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
155,17.63,15,1002,1952,254,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
104,31.96,10,165,1423,51,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
347,41.59,7,65,1606,390,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
86,28.72,20,130,145,342,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
75,10.08,3,1012,1855,181,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
438,21.63,20,1090,879,119,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [6]:
# GUARDAR MODELO
joblib.dump(model,"churn_model.pkl")
joblib.dump(imputer,"imputer.pkl")
joblib.dump(encoder,"encoder.pkl")
print("¡Los 3 archivos han sido guardados correctamente!")

¡Los 3 archivos han sido guardados correctamente!
